# StepForge Flow v1.5b — lean training on Colab

Trains the step network on human charts plus **counterfactual footing variants**
(the same rhythm stepped five ways, see `neural/synth/counterfactual.py`) and
the rhythm network on the human charts alone with half-beat-shifted crops.
No Demucs, no audio augmentation, no synthesized audio: about 25–45 minutes on
an A100, and the corpus to upload is the compiled human charts only.

**Before you start**, compile the human charts on the machine that holds the
song packs and copy the folder to Drive:

```bash
pip install -r requirements.txt -r requirements-train.txt
PYTHONPATH=tools python3 -m beatforge.adapters.stepmania.neural.train.corpus \
    --packs-dir <packs> --split-manifest <locked manifest.json> --out-dir build/v15b_corpus/real
tar -cf v15b_real.tar -C build/v15b_corpus real      # -> MyDrive/StepForge/v15b_upload/
```

Everything trains from the VM's local disk. Checkpoints, logs and manifests are
**mirrored to Drive as they are written** (`--mirror-dir`): `best.pt` whenever it
improves, `last.pt` every ten epochs, everything at the end of each phase. If
the VM is reclaimed, re-run the cells: cell 1 restores the run folders from
Drive and the trainers resume from `last.pt`.

Validation and checkpoint selection use the `val` split of `real/` only. The
counterfactual charts are built from training songs and carry their song ids;
the loader refuses a corpus where a derived item touches a validation song.

## 1 · Drive, code, GPU

In [ ]:
REPO   = "https://github.com/Sweet-Papa-Technologies/StepForge.git"
BRANCH = "main"
DRIVE  = "/content/drive/MyDrive/StepForge"
UPLOAD = f"{DRIVE}/v15b_upload"       # v15b_real.tar
MIRROR = f"{DRIVE}/v15b_runs"         # checkpoints + logs + manifests (survive a reclaimed VM)
CORPUS = "/content/v15b_corpus"       # local SSD: never train from Drive directly
RUNS   = "/content/v15b_runs"         # local SSD

import json, os, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(MIRROR, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)

if not os.path.isdir("/content/StepForge/.git"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO, "/content/StepForge"], check=True)
else:
    subprocess.run(["git", "-C", "/content/StepForge", "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/StepForge/requirements.txt",
                "-r", "/content/StepForge/requirements-train.txt", "onnxruntime"], check=True)
os.environ["PYTHONPATH"] = "/content/StepForge/tools"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch
assert torch.cuda.is_available(), "no GPU: Runtime > Change runtime type > A100 or L4"
AMP = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
print(torch.cuda.get_device_name(0), "| mixed precision:", AMP)
NEURAL = "beatforge.adapters.stepmania.neural"

def run(module, *args):
    """Run one of the package's command-line tools and stream its output."""
    cmd = [sys.executable, "-W", "ignore", "-m", f"{NEURAL}.{module}", *map(str, args)]
    print("$", " ".join(cmd[4:]), flush=True)
    subprocess.run(cmd, check=True)

def train(stage, name, *args):
    """Train into RUNS/name, mirrored to MIRROR/name; resume from the mirror on a fresh VM."""
    out, mirror = f"{RUNS}/{name}", f"{MIRROR}/{name}"
    if not os.path.exists(f"{out}/last.pt") and os.path.exists(f"{mirror}/last.pt"):
        os.makedirs(out, exist_ok=True)
        for f in os.listdir(mirror):
            shutil.copy(f"{mirror}/{f}", f"{out}/{f}")
        print("restored", name, "from Drive")
    run(f"train.train_stage_{stage}", "--corpus-root", CORPUS, "--out", out, "--mirror-dir", mirror, *args)

## 2 · Corpus: unpack the human charts, write the counterfactual variants

About a minute. For every Medium, Hard and Challenge chart of a training song,
and for 300 synthetic stream rhythms, up to five sets of arrows on the same
notes: clean alternation, crossover-heavy, footswitch-heavy, jack-heavy, and a
blend. Each is read back by the codec's annotator and labelled with what it
measures; a variant that is not legal, or not really apart from the clean one,
is dropped.

In [ ]:
if not os.path.exists(f"{CORPUS}/real/manifest.json"):
    os.makedirs(CORPUS, exist_ok=True)
    subprocess.run(["tar", "-xf", f"{UPLOAD}/v15b_real.tar", "-C", CORPUS], check=True)
if not os.path.exists(f"{CORPUS}/synth_cf/manifest.json"):
    run("synth.counterfactual", "--corpus-root", CORPUS, "--streams", 300, "--workers", 8)
from beatforge.adapters.stepmania.neural.synth import shards
print(shards.check_no_leak(CORPUS))
os.makedirs(f"{MIRROR}/manifests", exist_ok=True)
for src in sorted(os.listdir(CORPUS)):
    m = json.load(open(f"{CORPUS}/{src}/manifest.json"))
    print(f"{src:18s} {m['n_songs']:5d} shards  {m['n_charts_kept']:6d} charts  {m['splits']}")
    for f in ("manifest.json", "profile.json"):
        if os.path.exists(f"{CORPUS}/{src}/{f}"):
            shutil.copy(f"{CORPUS}/{src}/{f}", f"{MIRROR}/manifests/{src}_{f}")

## 3 · Stage B — ChoreoNet on counterfactual pairs

Phase 1 (85 epochs): human charts 45 % (audio withheld on 15 %), re-stepped
training songs 45 %, re-stepped stream rhythms 10 %. Phase 2 (40 epochs, low
learning rate): 65 / 30 / 5. Counterfactual charts are drawn in pairs — two
variants of one rhythm, same window — with their three footing sliders always
visible. Every epoch a probe samples held-out human rhythms with one slider
set low and then high; `best.pt` is the checkpoint with the best validation
cross-entropy among those whose sliders work, `best_ce.pt` the plain best.
About 25 minutes on an A100.

In [ ]:
train("b", "stage_b", "--phase1-epochs", 85, "--phase2-epochs", 40, "--batch", 64, "--amp", AMP,
      "--p-drop-audio", 0.15, "--p-drop-audio2", 0.15, "--p-drop-stems", 1.0,
      "--probe-every", 1, "--workers", 8)

## 4 · Stage A — RhythmNet, lean

The human charts only, with half-beat-shifted crops (what fixed the beat-phase
check in v1.5) and the 16th/triplet weights. `best.pt` is chosen inside the
low-learning-rate phase by `0.50 precision at the chart's own note count +
0.25 F1 on Hard/Challenge 16ths + 0.25 beat-phase accuracy`; `best_early.pt`
keeps the best v1.5 score of any epoch for comparison. Compare both on your
tuning songs before choosing. About 15 minutes on an A100.

In [ ]:
train("a", "stage_a", "--phase1-epochs", 65, "--phase2-epochs", 30, "--epoch-items", 950, "--warmup", 300,
      "--batch", 32, "--amp", AMP, "--dropout", 0.25, "--weight-decay", 0.03, "--p-drop-stems", 1.0,
      "--sub16-weight", 1.45, "--sub12-weight", 1.30, "--phase-margin-weight", 0,
      "--phase-shift-prob", 0.25, "--select", "polish_score", "--workers", 8)

## 5 · Export, parity check, phrase index

Refuses to write `model_meta.json` unless both graphs match PyTorch to 1e-4
and the step network's training-only "no audio" vector is absent from the
exported graph. The two networks are separate files with v1.4's inputs and
outputs, so a new step network can also be paired with an existing rhythm
network by copying files.

In [ ]:
MODEL = "/content/stepforge_flow_v15b"
run("train.export_onnx", "--stage-a", f"{RUNS}/stage_a/best.pt", "--stage-b", f"{RUNS}/stage_b/best.pt",
    "--out", MODEL, "--int8", "--version", "1.5")
run("style_rag", "--corpus", f"{CORPUS}/real", "--out", MODEL)
meta = json.load(open(f"{MODEL}/model_meta.json"))
print("parity:", meta["stage_a"]["parity_max_abs_diff"], meta["stage_b"]["parity_max_abs_diff"])
print("stage A val:", {k: round(v, 4) for k, v in meta["metrics"]["stage_a"].items()
                       if k in ("ap", "r_prec", "f1_hard16", "phase_acc", "polish_score")})
print("stage B val:", {k: round(v, 4) for k, v in meta["metrics"]["stage_b"].items()
                       if k in ("ce", "probe_sens", "probe_cross_delta", "probe_jack_delta", "probe_fsw_delta")})

## 6 · Package the release candidate

Weights and training logs, zipped to Drive next to the mirrored checkpoints.
Download it, calibrate the decoder on your tuning songs
(`python -m beatforge.adapters.stepmania.neural.calibrate_decoder`) and run
your benchmark before it replaces the shipped model.

In [ ]:
os.makedirs(f"{MODEL}/train", exist_ok=True)
for stage in "ab":
    for f in ("log.jsonl", "config.json"):
        shutil.copy(f"{RUNS}/stage_{stage}/{f}", f"{MODEL}/train/stage_{stage}_{f}")
for src in sorted(os.listdir(CORPUS)):
    shutil.copy(f"{CORPUS}/{src}/manifest.json", f"{MODEL}/train/manifest_{src}.json")
out = shutil.make_archive(f"{MIRROR}/v15b_release_candidate", "zip", MODEL)
print(out, round(os.path.getsize(out) / 1e6), "MB")
print("on Drive:", sorted(os.listdir(MIRROR)))